# Ward

Author: [Xin Feng](https://github.com/xf37), [James Gaboardi](https://github.com/jGaboardi)

This algorithm is an agglomerative clustering using ward linkage with a spatial connectivity constraint. Specifically, it is a "bottom-up" approach: each zone starts as its own cluster, and pairs of clusters are chosen to merge at each step in order to minimally increase a given linkage distance. Ward linkage refers to the variance of the clusters being merged. `Ward` algorithm in pysal/spopt is the function
([sklearn.cluster.AgglomerativeClustering](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html)) when the linkage criterion is ward. 

In [ ]:
%config InlineBackend.figure_format = "retina"
%load_ext watermark
%watermark

In [ ]:
import geopandas
import libpysal
from libpysal.examples import load_example

import spopt
from spopt.region import WardSpatial

%matplotlib inline
%watermark -w
%watermark -iv

## Airbnb Spots Clustering in Chicago

To illustrate `Ward` we utilize data on Airbnb spots in Chicago, which can be downloaded from [libpysal.examples](https://pysal.org/notebooks/lib/libpysal/Example_Datasets.html).

We can first explore the data by plotting the number of Airbnb spots in each community in the sample, using a quintile classification:

In [ ]:
load_example("AirBnB")

In [ ]:
pth = libpysal.examples.get_path("airbnb_Chicago 2015.shp")
chicago = geopandas.read_file(pth)
chicago.plot(
    figsize=(7, 14),
    column="num_spots",
    scheme="Quantiles",
    cmap="GnBu",
    edgecolor="grey",
    legend=True,
).axis("off");

## Regionalization

With `Ward`, we can aggregate these 77 communities into 5 clusters. During the merging process, the variance of the clusters is minimized.

We first define the variable that will be used to measure the variance of clusters. The variable is the number of Airbnb spots in each community in this case.

In [ ]:
attrs_name = ["num_spots"]

Next, we specify a number of other parameters that will serve as input to the `Ward` model.

A [spatial weights object](https://pysal.org/libpysal/tutorial.html) describes the spatial connectivity of the spatial objects:

In [ ]:
w = libpysal.weights.Queen.from_dataframe(chicago, use_index=False)

The number of clusters that we would like to group these counties into:

In [ ]:
n_clusters = 5

There are also some optional parameters about clustering in ([sklearn.cluster.AgglomerativeClustering](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html)). They can be added in the `Ward` function as a dictionary. In this example, we only use the default settings, you can define them as needed.

The model can then be solved:

In [ ]:
model = WardSpatial(chicago, w, attrs_name, n_clusters)
model.solve()

In [ ]:
chicago["ward_new"] = model.labels_

In [ ]:
chicago["number"] = 1
chicago[["ward_new", "number"]].groupby(by="ward_new").count()

In [ ]:
chicago.plot(figsize=(7, 14), column="ward_new", categorical=True, edgecolor="w").axis(
    "off"
);

The model solution results in five clusters, two of which have three communities, one with two, one with seven, and one with sixty-two communities. 